# Week 12: Capstone Project 5.2 — SQL Agent

This notebook demonstrates a working SQL agent that:
- Converts natural language queries into SQL using GPT-4
- Validates the SQL for safety
- Executes queries on a SQLite database (`company.db`)
- Formats the results into readable output

The system uses the OpenAI API and supports safe, read-only SQL execution via `SELECT` statements only.


In [1]:
# Keys are read from the environment - copy .env.example to .env and fill
# it in. Never hardcode credentials in a notebook.
import os

from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]


# Week 12: Capstone Project Part 5.2
## Natural Language to SQL Agent

**Objective:**  
Build an agent that converts natural language queries into SQL, executes them, and returns formatted results using SQLite and OpenAI function calling.

---
### ✅ Part 1: Setting Up the Database


In [2]:
import sqlite3

def setup_database():
    conn = sqlite3.connect('company.db')
    c = conn.cursor()

    # Create sample tables
    c.execute('''
        CREATE TABLE IF NOT EXISTS employees (
            id INTEGER PRIMARY KEY,
            name TEXT,
            department TEXT,
            salary REAL
        )
    ''')

    c.execute('''
        CREATE TABLE IF NOT EXISTS departments (
            id INTEGER PRIMARY KEY,
            name TEXT,
            budget REAL
        )
    ''')

    # Insert sample data
    c.execute("INSERT OR IGNORE INTO employees VALUES (1, 'John Doe', 'Engineering', 75000)")
    c.execute("INSERT OR IGNORE INTO employees VALUES (2, 'Jane Smith', 'Marketing', 65000)")
    c.execute("INSERT OR IGNORE INTO departments VALUES (1, 'Engineering', 1000000)")
    c.execute("INSERT OR IGNORE INTO departments VALUES (2, 'Marketing', 500000)")

    conn.commit()
    conn.close()

# Test setup
setup_database()

# Verify setup
conn = sqlite3.connect('company.db')
c = conn.cursor()

print("✅ Employees table:")
c.execute("SELECT * FROM employees")
print(c.fetchall())

print("\n✅ Departments table:")
c.execute("SELECT * FROM departments")
print(c.fetchall())

conn.close()


✅ Employees table:
[(1, 'John Doe', 'Engineering', 75000.0), (2, 'Jane Smith', 'Marketing', 65000.0)]

✅ Departments table:
[(1, 'Engineering', 1000000.0), (2, 'Marketing', 500000.0)]


## ✅ Part 2: Creating the SQL Generator

We implemented a function that uses OpenAI GPT-4 to convert natural language questions into valid SQL queries. The schema is passed as a system prompt to ensure accuracy, and the output is cleaned of markdown formatting.

### ✨ Features:
- Schema passed as context to GPT
- Sanitization of markdown and stray formatting
- Tested with simple employee and department queries


In [6]:
# Keys are read from the environment - copy .env.example to .env and fill
# it in. Never hardcode credentials in a notebook.
import os

from dotenv import load_dotenv

load_dotenv()

# Part 2: Creating the SQL Generator

import openai

# Direct API key (for Colab, not for production use!)
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

openai.api_key = OPENAI_API_KEY


# Define the database schema for GPT context
def get_schema():
    return """
    Table: employees
    Columns:
    - id (INTEGER PRIMARY KEY)
    - name (TEXT)
    - department (TEXT)
    - salary (REAL)

    Table: departments
    Columns:
    - id (INTEGER PRIMARY KEY)
    - name (TEXT)
    - budget (REAL)
    """

import openai

client = openai.OpenAI(api_key=OPENAI_API_KEY)

def generate_sql(question):
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": f"You are a SQL expert. Use this schema:\n{get_schema()}\nReturn ONLY the SQL query without any explanation or markdown formatting."},
            {"role": "user", "content": f"Generate SQL for: {question}"}
        ]
    )

    sql = response.choices[0].message.content.strip()

    # Sanitize: remove markdown artifacts
    sql = sql.replace("```sql", "").replace("```SQL", "").replace("```", "")
    sql_lines = [line.strip() for line in sql.split('\n') if line.strip()]
    return ' '.join(sql_lines)


# 🔍 Test SQL Generation
if __name__ == "__main__":
    test_questions = [
        "List all employees",
        "Show departments with budgets over 750000"
    ]

    for question in test_questions:
        print(f"\n🧠 Question: {question}")
        print(f"📝 Generated SQL: {generate_sql(question)}")



🧠 Question: List all employees
📝 Generated SQL: SELECT * FROM employees;

🧠 Question: Show departments with budgets over 750000
📝 Generated SQL: SELECT * FROM departments WHERE budget > 750000;


### 🚀 Part 3: Implementing Query Execution

Now that we can generate SQL queries from natural language, this part focuses on executing those queries against the SQLite database (`company.db`) and returning the results in a readable format.


In [8]:
def validate_sql(sql):
    sql_lower = sql.lower()
    if any(word in sql_lower for word in ['drop', 'delete', 'update', 'insert']):
        raise ValueError("Only SELECT queries are allowed")
    return sql


In [9]:
import sqlite3

# 🧠 Execute the generated SQL and return results
def execute_sql_query(query):
    try:
        conn = sqlite3.connect("company.db")
        c = conn.cursor()
        c.execute(query)
        results = c.fetchall()
        conn.close()
        return results
    except Exception as e:
        return f"Error executing query: {e}"

# 🧪 Test execution with natural language → SQL → results
test_questions = [
    "List all employees",
    "Show departments with budgets over 750000"
]

for question in test_questions:
    sql_query = generate_sql(question)
    results = execute_sql_query(sql_query)
    print(f"\n❓ Question: {question}")
    print(f"🧾 SQL: {sql_query}")
    print(f"📊 Results: {results}")
def format_results(results):
    if not isinstance(results, list):
        return str(results)
    if not results:
        return "No results found"

    if len(results[0]) == 1:
        return "\n".join([str(row[0]) for row in results])

    formatted_rows = []
    for row in results:
        row_items = []
        for item in row:
            if isinstance(item, float):
                row_items.append(f"${item:,.2f}" if "salary" in str(row) or "budget" in str(row) else f"{item:.2f}")
            else:
                row_items.append(str(item))
        formatted_rows.append("\t".join(row_items))
    return "\n".join(formatted_rows)



❓ Question: List all employees
🧾 SQL: SELECT * FROM employees;
📊 Results: [(1, 'John Doe', 'Engineering', 75000.0), (2, 'Jane Smith', 'Marketing', 65000.0)]

❓ Question: Show departments with budgets over 750000
🧾 SQL: SELECT * FROM departments WHERE budget > 750000;
📊 Results: [(1, 'Engineering', 1000000.0)]


In [11]:
def validate_sql(sql):
    sql_lower = sql.lower()
    if any(word in sql_lower for word in ['drop', 'delete', 'update', 'insert']):
        raise ValueError("Only SELECT queries are allowed")
    return sql

def execute_query(sql):
    sql = validate_sql(sql)
    conn = sqlite3.connect('company.db')
    try:
        cursor = conn.cursor()
        cursor.execute(sql)
        results = cursor.fetchall()
        return results
    except Exception as e:
        return f"Error: {str(e)}"
    finally:
        conn.close()

def format_results(results):
    if not isinstance(results, list):
        return str(results)
    if not results:
        return "No results found"
    if len(results[0]) == 1:
        return "\n".join([str(row[0]) for row in results])
    if isinstance(results[0], tuple):
        formatted_rows = []
        for row in results:
            row_items = []
            for item in row:
                if isinstance(item, float):
                    row_items.append(f"${item:,.2f}" if "salary" in str(row) or "budget" in str(row) else f"{item:.2f}")
                else:
                    row_items.append(str(item))
            formatted_rows.append("\t".join(row_items))
        return "\n".join(formatted_rows)
    return "\n".join([str(row) for row in results])


## Part 4: Creating the Complete Agent

In this section, we test the full pipeline — taking a natural language question, generating SQL using the OpenAI API, validating it for safety, executing it against the `company.db` database, and formatting the results for display. This also includes error handling (e.g., rejecting `DROP`, `DELETE`, etc.).


In [12]:
def query_agent(question):
    try:
        sql = generate_sql(question)
        print(f"Generated SQL: {sql}\n")
        results = execute_query(sql)
        return format_results(results)
    except Exception as e:
        return f"Error: {str(e)}"

# Test the complete agent
if __name__ == "__main__":
    test_questions = [
        "What is the average salary in each department?",
        "Which department has the highest budget?",
        "List all employees earning more than 70000",
        "DROP TABLE employees"  # Should be blocked
    ]

    for question in test_questions:
        print(f"\nQuestion: {question}")
        print(f"Answer: {query_agent(question)}")



Question: What is the average salary in each department?
Generated SQL: SELECT department, AVG(salary) FROM employees GROUP BY department;

Answer: Engineering	75000.00
Marketing	65000.00

Question: Which department has the highest budget?
Generated SQL: SELECT name FROM departments ORDER BY budget DESC LIMIT 1;

Answer: Engineering

Question: List all employees earning more than 70000
Generated SQL: SELECT * FROM employees WHERE salary > 70000;

Answer: 1	John Doe	Engineering	75000.00

Question: DROP TABLE employees
Generated SQL: DROP TABLE employees;

Answer: Error: Only SELECT queries are allowed


## ✅ SQL Agent Summary and Error Handling Overview

This project demonstrates a basic SQL-generating agent using OpenAI's GPT model and SQLite. It translates natural language queries into SQL, executes them safely, and formats the result for clarity.

### 🔧 Components Implemented
- **SQL Generation**: `generate_sql()` uses the OpenAI API with a clearly defined schema.
- **SQL Validation**: `validate_sql()` ensures only safe `SELECT` queries are executed. Malicious queries like `DROP` or `DELETE` are blocked.
- **Query Execution**: `execute_query()` connects to `company.db`, runs validated SQL, and returns results.
- **Result Formatting**: `format_results()` prints tables neatly with currency formatting for salaries and budgets.

### 📊 Example Tests Run
- “What is the average salary in each department?”  
- “Which department has the highest budget?”  
- “List all employees earning more than 70000”  
- 🔐 Malicious Input: `DROP TABLE employees`  
  **Handled Response:** `Error: Only SELECT queries are allowed`

### 🧯 Error Handling Summary
- Queries containing `DROP`, `DELETE`, `UPDATE`, or `INSERT` are blocked with a descriptive error message.
- SQL errors during execution are caught with `try/except`, preventing crashes.
- Unexpected results are handled with checks for empty datasets and data types.

### 🧰 Troubleshooting Notes
- If you get `"Error: Only SELECT queries are allowed"` → Validation is working.
- If the query is malformed → Output will return a caught exception.
- If OpenAI returns multiple lines or markdown → Code strips extraneous formatting.

---

